# 🏆 Hackathon Results
This notebook connects to your live MongoDB database to fetch all vote records. It manually calculates the final score by applying the 2.5x multiplier for `judge26` and the 1x multiplier for `victorhacks`.

In [ ]:
# Install necessary packages directly into your Jupyter environment
# %pip install pymongo pandas matplotlib python-dotenv

In [ ]:
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
import json

In [ ]:
# Load MONGODB_URI from .env
load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')

if not MONGODB_URI:
    print("Error: MONGODB_URI not found in .env file!")
else:
    print("Connecting to database...")
    client = pymongo.MongoClient(MONGODB_URI)
    db = client['test']
    print("✅ Connected to MongoDB!")

In [ ]:
# 1. Load project definitions to map IDs to Names
with open('projects.json', 'r') as f:
    projects_list = json.load(f)

projects_dict = {p['id']: p['name'] for p in projects_list}

# 2. Define the exact valid codes and their weights
CODE_WEIGHTS = {
    'victorhacks': 1.0,
    'judge26': 2.5
}

# 3. Fetch raw vote records
votes_col = db['voterecordv2s']
raw_votes = list(votes_col.find({}))

# 4. Calculate total weighted score from scratch
# We do this from scratch so any old test votes with different codes are ignored!
project_totals = {pid: 0 for pid in projects_dict.keys()}

for vote in raw_votes:
    code = vote.get('code', '').lower()
    pid = vote.get('projectId')
    score = vote.get('score', 0)
    
    if code in CODE_WEIGHTS and pid in project_totals:
        weight = CODE_WEIGHTS[code]
        project_totals[pid] += (score * weight)

df = pd.DataFrame(list(project_totals.items()), columns=['projectId', 'votes'])
df['Project Name'] = df['projectId'].map(projects_dict)

# Sort descending
df = df.sort_values(by='votes', ascending=False).reset_index(drop=True)
display(df)

### 🥇 Announcing the Winner

In [ ]:
# Filter out ineligible projects (e.g. CarbonCredible PROJ-002)
ineligible_ids = ['PROJ-002']
eligible_df = df[~df['projectId'].isin(ineligible_ids)].reset_index(drop=True)

if not eligible_df.empty:
    winner = eligible_df.iloc[0]
    print("\n" + "*"*40)
    print(f"🏆 THE WINNER IS: {winner['Project Name']}!")
    print(f"Total Weighted Score: {winner['votes']}")
    print("*"*40 + "\n")
    
    print("Runner Ups:")
    display(eligible_df.iloc[1:4])

### 📊 Visualization

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(df['Project Name'], df['votes'], color='#667eea')
plt.xlabel('Projects', fontsize=12)
plt.ylabel('Total Weighted Score', fontsize=12)
plt.title('Final Hackathon Voting Results', fontsize=16)
plt.xticks(rotation=45, ha='right')

# Add values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, yval, ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()